In [ ]:
!pip install torchinfo

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import os
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchinfo import summary

In [ ]:
# ----------------------------
# Device selection
# ----------------------------
if torch.cuda.is_available():
    device = torch.device("cuda")
elif 'TPU_ACCELERATOR_TYPE' in os.environ or 'COLAB_TPU_1vm' in os.environ:
    import torch_xla
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
# Preprocessing: scale to [0, 1] for VAE (sigmoid output), flatten later
transform_vae = transforms.Compose([
    transforms.ToTensor(),  # [0,1]
])

train_dataset_vae = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform_vae
)
train_loader_vae = DataLoader(train_dataset_vae, batch_size=128, shuffle=True, num_workers=2)

# Also keep original [-1,1] loader for comparison (optional)
transform_ssm = transforms.Compose([
    transforms.Pad(2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    lambda x: x * 2 - 1
])
train_dataset_ssm = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform_ssm
)
train_loader_ssm = DataLoader(train_dataset_ssm, batch_size=128, shuffle=True, num_workers=2)

In [ ]:
# ----------------------------
# VAE Components (as provided)
# ----------------------------
def reparameterize(mu, log_var):
    std = torch.exp(0.5 * log_var)
    eps = torch.randn_like(std)
    return mu + eps * std

class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, z_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, z_dim)
        self.fc_logvar = nn.Linear(hidden_dim, z_dim)
    def forward(self, x):
        h = F.relu(self.fc1(x))
        mu = self.fc_mu(h)
        log_var = self.fc_logvar(h)
        return mu, log_var

class Decoder(nn.Module):
    def __init__(self, z_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(z_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
    def forward(self, z):
        h = F.relu(self.fc1(z))
        x_recon = torch.sigmoid(self.fc_out(h))
        return x_recon

class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, z_dim):
        super().__init__()
        self.encoder = Encoder(input_dim, hidden_dim, z_dim)
        self.decoder = Decoder(z_dim, hidden_dim, input_dim)
    def forward(self, x):
        mu, log_var = self.encoder(x)
        z = reparameterize(mu, log_var)
        x_recon = self.decoder(z)
        return x_recon, mu, log_var

In [ ]:
# Loss function
# ELBO = Reconstruction loss + KL divergence
#
def vae_loss(x, x_recon, mu, log_var):
    # Fixed variance sigma^2 = 0.5 (sigma = sqrt(2)/2)
    # For sigma^2=0.5, 2*sigma^2=1, so denominator is 1
    # Gaussian NLL (ignoring constant term) reduces to MSE sum
    recon_loss = F.mse_loss(x_recon, x, reduction="sum")
    # KL divergence between q(z|x) and p(z)
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return recon_loss + kl_loss


In [ ]:
# ----------------------------
# Latent Score Network
# ----------------------------
class LatentScoreNet(nn.Module):
    def __init__(self, z_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, z_dim)
        )
    def forward(self, z):
        return self.net(z)


In [ ]:

# ----------------------------
# Latent-space SSM-VR Loss
# ----------------------------
def ssm_vr_loss_latent(score_net, z, n_projections=2):
    """
    SSM-VR in latent space (z is [B, D])
    """
    B, D = z.shape
    z = z.requires_grad_(True)
    s = score_net(z)  # [B, D]

    v_hessian_v_total = 0.0
    norm_s_sq_total = 0.0
    for _ in range(n_projections):
        v = torch.randint(0, 2, (B, D), device=z.device, dtype=torch.float32) * 2 - 1
        v_dot_s = (v * s).sum(dim=1)
        grad_vdot_s = torch.autograd.grad(v_dot_s.sum(), z, create_graph=True)[0]
        v_hessian_v = (v * grad_vdot_s).sum(dim=1)
        v_hessian_v_total += v_hessian_v
        norm_s_sq = (s ** 2).sum(dim=1)
        norm_s_sq_total += norm_s_sq

    v_hessian_v_avg = v_hessian_v_total / n_projections
    norm_s_sq_avg = norm_s_sq_total / n_projections
    loss = (v_hessian_v_avg + 0.5 * norm_s_sq_avg).mean()
    return loss


In [ ]:

# ----------------------------
# Hyperparameters
# ----------------------------
input_dim = 28 * 28
hidden_dim = 512
z_dim = 32  # latent dimension
epochs_vae = 20
epochs_ssm = 30
lr_vae = 1e-3
lr_ssm = 1e-4


In [ ]:

# ----------------------------
# Stage 1: Train VAE
# ----------------------------
print("=== Stage 1: Training VAE ===")
vae = VAE(input_dim, hidden_dim, z_dim).to(device)
vae_optimizer = torch.optim.Adam(vae.parameters(), lr=lr_vae)

vae.train()


In [ ]:

for epoch in range(epochs_vae):
    total_loss = 0.0
    for batch_idx, (x, _) in enumerate(train_loader_vae):
        x = x.view(x.size(0), -1).to(device)
        vae_optimizer.zero_grad()
        x_recon, mu, log_var = vae(x)
        loss = vae_loss(x, x_recon, mu, log_var)
        loss.backward()
        vae_optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader_vae.dataset)
    print(f"VAE Epoch {epoch+1}/{epochs_vae} | Avg ELBO per sample: {avg_loss:.4f}")


In [ ]:

# Save VAE
torch.save(vae.state_dict(), "vae_mnist.pth")
print("VAE trained and saved.")


In [ ]:

# ----------------------------
# Stage 2: Train Latent Score Model
# ----------------------------
print("\n=== Stage 2: Training Latent Score Model ===")
# Freeze VAE
for param in vae.parameters():
    param.requires_grad_(False)
vae.eval()

score_net = LatentScoreNet(z_dim).to(device)
score_optimizer = torch.optim.Adam(score_net.parameters(), lr=lr_ssm)
scheduler_ssm = torch.optim.lr_scheduler.ReduceLROnPlateau(score_optimizer, mode='min', factor=0.5, patience=2, verbose=True)

# Collect latent samples from training set (one-time or on-the-fly)
# We'll do on-the-fly for memory efficiency
score_net.train()


In [ ]:

for epoch in range(epochs_ssm):
    total_loss = 0.0
    num_batches = 0
    for x, _ in train_loader_vae:
        x = x.view(x.size(0), -1).to(device)
        with torch.no_grad():
            mu, log_var = vae.encoder(x)
            z = reparameterize(mu, log_var)  # samples from q(z|x)
        z = z.detach().requires_grad_(True)  # stop grad from VAE, enable for SSM

        score_optimizer.zero_grad()
        loss = ssm_vr_loss_latent(score_net, z, n_projections=32)
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"NaN/Inf loss at epoch {epoch+1}, batch {num_batches}")
            break
        loss.backward()
        score_optimizer.step()
        total_loss += loss.item()
        num_batches += 1

    if num_batches == 0:
        break
    avg_loss = total_loss / num_batches
    print(f"SSM Epoch {epoch+1}/{epochs_ssm} | Avg Loss: {avg_loss:.6f} | LR: {score_optimizer.param_groups[0]['lr']:.2e}")
    scheduler_ssm.step(avg_loss)


In [ ]:

torch.save(score_net.state_dict(), "latent_score_net.pth")
print("Latent score model trained and saved.")


In [ ]:
# ----------------------------
# Sampling: Langevin in Latent Space + Decode
# ----------------------------
@torch.no_grad()
def langevin_latent(score_net, vae_decoder, shape, n_steps=1000, eps=1e-3, device="cpu"):
    """
    shape: (B,)
    Returns: decoded samples [B, 1, 28, 28] in [0,1]
    """
    z = torch.randn(shape, device=device)
    score_net.eval()
    vae_decoder.eval()
    for _ in range(n_steps):
        z = z.requires_grad_(True)
        s = score_net(z)  # [B, z_dim]
        noise = torch.randn_like(z) * (2 * eps) ** 0.5
        z = z + eps * s + noise
        z = z.detach()
    x_recon = vae_decoder(z)
    return x_recon.view(-1, 1, 28, 28)


In [ ]:

# Generate samples
print("\n=== Generating Samples ===")
samples = langevin_latent(score_net, vae.decoder, (64, z_dim), n_steps=1000, eps=0.001, device=device)

# Plot
plt.figure(figsize=(8, 8))
grid = torchvision.utils.make_grid(samples, nrow=8, padding=2)
plt.imshow(grid.cpu().permute(1, 2, 0).numpy(), cmap='gray')
plt.axis('off')
plt.title("Generated MNIST (Latent SSM + VAE Decoder)")
plt.savefig("latent_ssm_samples.png", dpi=150, bbox_inches='tight')
plt.show()

print("Done!")